# add-sub-div-back-lambdas — ex1: build BACK dict — 6 lambdas for add/sub/div × arg0/arg1

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `add-sub-div-back-lambdas`. Running the final beacon cell reports progress against the `Backprop: add/sub/div back as lambdas` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: add/sub/div back as lambdas` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`add-sub-div-back-lambdas`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "add-sub-div-back-lambdas"
DD_SUBTOPIC = "Backprop: add/sub/div back as lambdas"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `add` / `sub` / `div` back as lambdas — quick refresher

When two backward fns are short one-liners with the same `(grad_out, out, x, y)` signature, define them inline as lambdas keyed by `(op, argnum)` — no `def` boilerplate, no per-fn name to remember.

**Worked exemplar.** For `out = x - y`:
```
d(x - y)/dx =  1   →  sub_back0 = lambda g, o, x, y:  g
d(x - y)/dy = -1   →  sub_back1 = lambda g, o, x, y: -g
```

Use a dict to store all 6 lambdas (3 ops × 2 argnums) so the dispatcher can look one up with `BACK[(op, argnum)](grad_out, out, x, y)`.

### Exercise 1 — build BACK dict — 6 lambdas for add/sub/div × arg0/arg1

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the lambda-keyed back-fn dispatch pattern: build a dict mapping (op_name, argnum) → backward lambda for add, sub, div.
> Keywords: lambda, back-dict, add, sub, div, arg-position
> ```

**KCs targeted:** `arg-position-back-functions`, `backward-fn-signature`

Build a single dict `BACK` mapping `(op_name, argnum)` to a backward lambda. Six entries total: `add`, `sub`, `div` × argnum `0`, `1`.

Each lambda has signature `(grad_out, out, x, y) -> Tensor` and returns `dL/dx` for argnum=0 or `dL/dy` for argnum=1.

Derivations (no broadcasting in this drill — assume `x.shape == y.shape == out.shape`):

```
out = x + y:
  d/dx =  1   →  BACK[('add', 0)] = lambda g, o, x, y:  g
  d/dy =  1   →  BACK[('add', 1)] = lambda g, o, x, y:  g

out = x - y:
  d/dx =  1   →  BACK[('sub', 0)] = lambda g, o, x, y:  g
  d/dy = -1   →  BACK[('sub', 1)] = lambda g, o, x, y: -g

out = x / y:
  d/dx =  1/y      →  BACK[('div', 0)] = lambda g, o, x, y:  g / y
  d/dy = -x/y**2   →  BACK[('div', 1)] = lambda g, o, x, y: -g * x / (y * y)
```

**Why lambdas, not `def`.** Each body is a one-liner; the def boilerplate would be louder than the math. Storing them in a dict by `(op, argnum)` also matches how the autograd dispatcher actually looks back fns up at reverse time.

Build `BACK` as a module-level dict. No autograd.

In [ ]:
# Build BACK: dict[(str, int), Callable]
# Keys: ('add', 0), ('add', 1), ('sub', 0), ('sub', 1), ('div', 0), ('div', 1)
# Each value is a lambda(grad_out, out, x, y) -> Tensor.
BACK = {
    # fill me in
}


def _test_ex1():
    # --- key set is exactly the 6 expected pairs ---
    expected_keys = {('add', 0), ('add', 1), ('sub', 0), ('sub', 1), ('div', 0), ('div', 1)}
    assert set(BACK.keys()) == expected_keys, (
        f'BACK keys mismatch: extra={set(BACK.keys()) - expected_keys}, '
        f'missing={expected_keys - set(BACK.keys())}'
    )

    # --- all values are callable ---
    for k, v in BACK.items():
        assert callable(v), f'BACK[{k}] is not callable'

    # --- add ---
    x = t.tensor([1.0, 2.0, 3.0])
    y = t.tensor([4.0, 5.0, 6.0])
    out = x + y
    g_in = t.tensor([7.0, 8.0, 9.0])
    assert t.allclose(BACK[('add', 0)](g_in, out, x, y), g_in)
    assert t.allclose(BACK[('add', 1)](g_in, out, x, y), g_in)

    # --- sub ---
    out = x - y
    assert t.allclose(BACK[('sub', 0)](g_in, out, x, y),  g_in)
    assert t.allclose(BACK[('sub', 1)](g_in, out, x, y), -g_in)

    # --- div ---
    x = t.tensor([6.0, 8.0, 10.0])
    y = t.tensor([2.0, 4.0, 5.0])
    out = x / y
    g_in = t.tensor([1.0, 1.0, 1.0])
    g0 = BACK[('div', 0)](g_in, out, x, y)
    g1 = BACK[('div', 1)](g_in, out, x, y)
    assert t.allclose(g0, 1 / y)
    assert t.allclose(g1, -x / (y * y))

    # --- div: non-unit grad_out, witnessed by autograd ---
    x_ref = t.tensor([3.0, 5.0, 7.0], requires_grad=True)
    y_ref = t.tensor([2.0, 4.0, 6.0], requires_grad=True)
    z = (x_ref / y_ref).sum()
    z.backward()
    x_det, y_det = x_ref.detach(), y_ref.detach()
    out_cached = x_det / y_det
    g0_ours = BACK[('div', 0)](t.ones(3), out_cached, x_det, y_det)
    g1_ours = BACK[('div', 1)](t.ones(3), out_cached, x_det, y_det)
    assert t.allclose(g0_ours, x_ref.grad, atol=1e-6)
    assert t.allclose(g1_ours, y_ref.grad, atol=1e-6)

    # --- asymmetric ops must produce DIFFERENT results at the two argnums ---
    x_t = t.tensor([3.0, 5.0])
    y_t = t.tensor([2.0, 4.0])
    g_t = t.ones(2)
    assert not t.allclose(
        BACK[('sub', 0)](g_t, x_t - y_t, x_t, y_t),
        BACK[('sub', 1)](g_t, x_t - y_t, x_t, y_t),
    ), 'sub back0 and back1 must differ'
    assert not t.allclose(
        BACK[('div', 0)](g_t, x_t / y_t, x_t, y_t),
        BACK[('div', 1)](g_t, x_t / y_t, x_t, y_t),
    ), 'div back0 and back1 must differ'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
BACK = {
    ('add', 0): lambda g, o, x, y:  g,
    ('add', 1): lambda g, o, x, y:  g,
    ('sub', 0): lambda g, o, x, y:  g,
    ('sub', 1): lambda g, o, x, y: -g,
    ('div', 0): lambda g, o, x, y:  g / y,
    ('div', 1): lambda g, o, x, y: -g * x / (y * y),
}
```

**Symmetric ops still get two entries.** `('add', 0)` and `('add', 1)` have identical bodies. The dispatcher doesn't know — it just looks up `(op, argnum)` and calls. Skip the second entry and `add(x, y).backward()` fails to flow grad to `y`.

**Lambdas vs `def`.** For genuinely one-line bodies, lambdas are fine. For anything with broadcasting (`unbroadcast` wrappers, conditional dim handling), promote to a named `def` — the dispatch key stays the same.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()